# CIS 5450 Final Project
## Predicting New York Airbnb Prices Using Airbnb Open Data

### Import libraries

In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# import plotly.express as px
import seaborn as sns
from collections import Counter
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor

In [16]:
df1 = pd.read_csv('dataset/2024-6-nyc.csv')
df2 = pd.read_csv('dataset/2024-6-jc.csv')

### Load the dataset

In [21]:
#Read Dataset
org_df = pd.concat([df1,df2])
org_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 39327 entries, 0 to 1725
Data columns (total 75 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            39327 non-null  int64  
 1   listing_url                                   39327 non-null  object 
 2   scrape_id                                     39327 non-null  int64  
 3   last_scraped                                  39327 non-null  object 
 4   source                                        39327 non-null  object 
 5   name                                          39325 non-null  object 
 6   description                                   38148 non-null  object 
 7   neighborhood_overview                         22220 non-null  object 
 8   picture_url                                   39326 non-null  object 
 9   host_id                                       39327 non-null  int64

### Data Preprocessing
1. Drop duplicates data
2. Drop Unimportant features for training
3. Convert column "price" and "service fee" to float data type
4. Correct basic logistic problems in column "availability 365" and "minimum nights"
5. Correct the typo in column "neighbourhood"
6. Drop null values

In [22]:
# #Drop duplicates data
# org_df.drop_duplicates(inplace=True)

# #Drop unimportant feature
cl_drop = ['id','name','host_id','host_name','last_review','reviews_per_month','calendar_updated', 'license', 'neighbourhood','latitude','longitude']
org_df.drop(columns=cl_drop, inplace=True)

# #Convert Price to float type
org_df['price'] = org_df['price'].replace('[\$,]', '', regex=True).astype(float)
# # org_df['service_fee'] = org_df['service_fee'].replace('[\$,]', '', regex=True).astype(float)

# #Remove availabile days less than 0 
# org_df['availability_365'] = np.where(org_df['availability_365']<0, org_df['availability_365']*-1, org_df['availability_365'])
# #Remove availabile days more than 365
# org_df['availability_365'] = np.where(org_df['availability_365']>365, 365, org_df['availability_365'])

# #Remove minimum nights less than 0 
# org_df['minimum_nights'] = np.where(org_df['minimum_nights']<0, org_df['minimum_nights']*-1, org_df['minimum_nights'])

# #Remove typo in neighbourhood
# # org_df.rename(columns = {'neighbourhood group':'neighbourhood_group'}, inplace = True)
# # org_df = org_df[org_df.neighbourhood_group != 'brookln']

# # Drop null values
# # final_df = org_df.dropna()

# final_df.info()

In [24]:
final_df = org_df.select_dtypes(exclude=['object'])
final_df = final_df.apply(lambda col: col.fillna(col.mean()), axis=0)
final_df.dropna(how='any')
final_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 39327 entries, 0 to 1725
Data columns (total 34 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   scrape_id                                     39327 non-null  int64  
 1   host_listings_count                           39327 non-null  float64
 2   host_total_listings_count                     39327 non-null  float64
 3   accommodates                                  39327 non-null  int64  
 4   bathrooms                                     39327 non-null  float64
 5   bedrooms                                      39327 non-null  float64
 6   beds                                          39327 non-null  float64
 7   price                                         39327 non-null  float64
 8   minimum_nights                                39327 non-null  int64  
 9   maximum_nights                                39327 non-null  int64

In [25]:
final_df.columns

Index(['scrape_id', 'host_listings_count', 'host_total_listings_count',
       'accommodates', 'bathrooms', 'bedrooms', 'beds', 'price',
       'minimum_nights', 'maximum_nights', 'minimum_minimum_nights',
       'maximum_minimum_nights', 'minimum_maximum_nights',
       'maximum_maximum_nights', 'minimum_nights_avg_ntm',
       'maximum_nights_avg_ntm', 'availability_30', 'availability_60',
       'availability_90', 'availability_365', 'number_of_reviews',
       'number_of_reviews_ltm', 'number_of_reviews_l30d',
       'review_scores_rating', 'review_scores_accuracy',
       'review_scores_cleanliness', 'review_scores_checkin',
       'review_scores_communication', 'review_scores_location',
       'review_scores_value', 'calculated_host_listings_count',
       'calculated_host_listings_count_entire_homes',
       'calculated_host_listings_count_private_rooms',
       'calculated_host_listings_count_shared_rooms'],
      dtype='object')

### Train and Test set split

In [26]:
X = final_df.drop(columns=['price'], axis=1).values
y = final_df['price'].values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [27]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
y_pred = lr.predict(X_test_scaled)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred) *100
print(mse,r2)

1384064.5560653375 0.8319113628766694


In [ ]:
# model = RandomForestRegressor(n_estimators=100, random_state=42)
# model.fit(X_train,y_train)

# importance = model.feature_importances_

# feature_importance_df = pd.DataFrame({
#     'Feature': [f'Feature {i+1}' for i in range(X.shape[1])],
#     'Importance': importance
# })

# feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# plt.figure(figsize=(8, 6))
# plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'])
# plt.xlabel('Feature Importance')
# plt.title('Feature Importance from Random Forest Model')
# plt.show()